In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from eucare.base import *
import eucare.plotting as euplot


plt.figure(figsize=(5, 5))
prev_poly = regular_poly_points(3)
for i in range(3, 20):
    poly = regular_poly_points(i) * np.random.rand()
    j = np.random.randint(0, i-2)
    mat = find_affine(poly[j:j+2][::-1], prev_poly[i-3:i-1])
    poly = apply_affine(poly, mat)
    euplot.plot_polygon(poly)
    prev_poly=poly
    #plt.scatter(*(poly[j:j+2].T))
    
euplot.set_equal_aspect()

## HalfEdge Data Structure

In [ ]:
from eucare.half import AttributeObject

a = AttributeObject()

print(a.has_attributes())
a['adsf'] = 'party'
print(a.has_attributes())
for key, val in a.items():
    print(key, val)


In [ ]:
from eucare.half import Vertex, CyclicHalfedgeGraph, IdObject
from tqdm import tqdm_notebook as tqdm
import networkx as nx

IdObject.reset_ids()
poly = CyclicHalfedgeGraph([Vertex() for i in range(4)])
#for v in poly.vertices:
#    print(v)
#    for h in v.outgoing_iter():
#        pass
#        print(h.orig, h.dest)

#hs = list(poly.halfedges)

#f = any_element(poly.faces)
#print(f.__dict__)
#[print(v) for v in f.reverse_halfedge_iter()]


for i in range(2):
    print(i, poly.order)
    border_vertices = poly.border_vertices()
    for h1 in tqdm(list(poly.border_edge_iter())):
        #print(h1)
        to_attach = CyclicHalfedgeGraph([Vertex() for i in range(5)])
        h2 = to_attach.get_any_border()
        poly.add_graph(to_attach)
        poly.glue_e2e(h1, h2)
    for v in border_vertices:
        poly.close_vertex(v)

G = poly.to_networkx_undirected()
pos = nx.spring_layout(G.to_undirected())
nx.draw_networkx_nodes(G, pos, cmap=plt.get_cmap('jet'), node_size=500)
nx.draw_networkx_edges(G, pos, edge_color='r', arrows=True)
nx.draw_networkx_labels(G, pos)
plt.show()

heg = EHEG_from_layout(G, pos)
heg.check_consistency()

for v in G.nodes:
    v['pos'] = pos[v]
    
rend = CairoRenderer(width=1500, scale=500, face_inset=0.02, line_width=0.02)
surface = rend.render_graph(heg)
filename = 'output.png'
surface.write_to_png(filename)
surface.finish()
img = mpimg.imread(filename)
plt.figure(figsize=(13, 8))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
from eucare.half import EuclideanPositionHEG, HalfEdge, Face, rotate_by
from eucare.base import angle_to_axis
from eucare.redering import CairoRenderer
import numpy as np
from copy import copy
import matplotlib.pyplot as plt


    
#for v in G.nodes:
    #print(v, G[v])
    
def signed_area(pts):
    pts = np.array(pts)
    assert pts.shape[1] == 2
    pts_rot = np.concatenate([pts[1:], pts[:1]])
    return np.sum(pts[:, 0] * pts_rot[:, 1] - pts[:, 1] * pts_rot[:, 0]) / 2

def EHEG_from_edgelist(pts, edges):
    pass

def EHEG_from_layout(nxg, positions=None):
    assert not nxg.is_directed()
    if positions is None:
        positions = {n:np.array(n) for n in nxg.nodes()}
    assert isinstance(positions, dict)
    result = EuclideanPositionHEG()
    v_lookup = dict()
    for n in G.nodes():
        v_lookup[n] = Vertex()
        v_lookup[n]['pos'] = positions[n]
    result.add_vertices(v_lookup.values())
    h_lookup = dict()
    # orig, dest
    for n in G.nodes:
        v = v_lookup[n]
        h_lookup[v] = dict()
        for m in G[n]:
            w = v_lookup[m]
            h_lookup[v][w] = HalfEdge(orig=v, dest=w)
            v.any_outgoing = h_lookup[v][w]
        result.add_halfedges(h_lookup[v].values())
    # rev
    for v in h_lookup:
        for w in h_lookup[v]:
            h_lookup[v][w].rev = h_lookup[w][v]
    # nex, pre
    for v in h_lookup:
        outgoing_halfedges = list(h_lookup[v].values())
        dirs = np.array([v['pos'] - h.dest['pos'] for h in outgoing_halfedges])
        angles = angle_to_axis(dirs) % (2 * np.pi)
        order = np.argsort(angles)
        outgoing_halfedges = [outgoing_halfedges[i] for i in order]
        for hrevnex, h, hprerev in rotate_by(outgoing_halfedges, (0, 1, 2)):
            h.rev.nex = hrevnex
            h.pre = hprerev.rev
    
    # the faces
    unassigned_edges = copy(result.halfedges)
    while unassigned_edges:
        h = next(iter(unassigned_edges))
        f = Face(any_side=h)
        result.add_face(f)
        for k in f.halfedge_iter():
            k.face = f
            unassigned_edges.remove(k)
    
    # detect 'outside' faces which should be None by their orientation
    for f in frozenset(result.faces):
        vertex_pos = [v['pos'] for v in f.vertex_iter()]
        if signed_area(vertex_pos) < 0:
            result.delete_face(f)
            
    return result



In [ ]:
def truncate_graph(t=1/2):
    v1 = (0, -1)
    vf = (1, 0)
    v2 = (0, 1)
    v12t = (0, -1+t)
    v1ft = (t/2, -1+t/2)
    v21t = (0, 1-t)
    v2ft = (t/2, 1-t/2)
    G = nx.Graph()
    G.add_cycle([v1, v1ft, vf, v2ft, v2, v21t, v12t], delete=True)
    G[v12t][v21t]['delete'] = False
    print('asdf', G[v12t])
    #G.add_nodes_from()
    G.add_edges_from([[v12t, v1ft], [v21t, v2ft]])
    # almost everything is deleted
    return G

G = truncate_graph()
for e in G.edges():
    print(e)
    print(G[e[0]][e[1]])

heg = EHEG_from_layout(G)
heg.check_consistency()

rend = CairoRenderer(width=1500, scale=300, face_inset=0.02, line_width=0.02)
surface = rend.render_graph(heg)
filename = 'output.png'
surface.write_to_png(filename)
surface.finish()
img = mpimg.imread(filename)
plt.figure(figsize=(13, 8))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
from eucare.half import HalfEdgeGraph, Vertex, IdObject, RegularNGon, CyclicHalfedgeGraph, InAngleHEG, EuclideanPositionHEG
from eucare.instructions import *
from eucare.base import unit_vector
from copy import deepcopy, copy
from tqdm import tqdm_notebook as tqdm
from eucare.plotting import plot_polygon
import matplotlib.pyplot as plt

    
#tile = copy(tile)

#def RegularNGon(n):
#    return CyclicHalfedgeGraph([Vertex() for _ in range(n)])

#n = 6
#proto_tile = RegularEuclideanTile(n, edge_labels = ['a'] * n)

def print_graph(graph):
    for e in graph.halfedges:
        print(e.__repr__(), e.on_border(), e.face)
        
def attatch_tile_instruction(proto_tile, label=None):
    def instruction(graph, edge):
        tile, edge_dict = proto_tile.make_graph()
        if label is not None:
            edge = edge_dict[label]
        else:
            # just take any edge
            edge = next(iter(edge_dict.values()))
        graph.glue_graph_e2e(tile, e, edge)
    return instruction

# define 6.4.3.4 tiling
hexagon = RegularEuclideanTile(6, edge_labels=['a', 'a', 'a', 'a', 'a', 'a'])
square = RegularEuclideanTile(4, edge_labels=['b', 'c', 'b', 'c'])
triangle = RegularEuclideanTile(3, edge_labels=['d', 'd', 'd'])
hexagon.edge_instructions['a'] = attatch_tile_instruction(square, 'b')
square.edge_instructions['b'] = attatch_tile_instruction(hexagon)
square.edge_instructions['c'] = attatch_tile_instruction(triangle)
triangle.edge_instructions['d'] = attatch_tile_instruction(square, 'c')
        

#tile = RegularNGon(n)
#print_graph(tile)
#for e in tile.border_edge_iter():
#    e['instruction'] = attatch_tile_instruction(proto_tile, 'a')
    

#for e in tile1.border_edge_iter():
#    e['instruction'] = instruction0
  

# possible problem: topology based merging can miss geometry

for k in tqdm(range(1)):
    IdObject.reset_ids()
    tiling = EuclideanPositionHEG(eps=1e-3, other=hexagon.make_graph(add_positions=True)[0])
    for i in range(1):
        for e in tiling.border_edges():
            if e.on_border() and e in tiling.halfedges:
                tiling.execute_edge_instruction(e)
                    
                #print('...',tiling.order,'...')

print('drawing layout')

for face in tiling.faces:
    points = np.stack([v['pos'] for v in face.vertex_iter()])
    plot_polygon(points)
plt.show()
#tiling.show_spring_layout()

In [ ]:
from eucare.base import unit_vector, angle_to_axis
unit_vector(np.pi/2)

In [ ]:
(8.37758 - 6.28318) / np.pi

In [ ]:
import numpy as np
import collections

class EuclideanVertex2D(Vertex):
    def __init__(self, pos, any_outgoing=None):
        super(EuclideanVertex2D, self).__init__(any_outgoing)
        if not isinstance(pos, colledctions.Sized):
            raise ValueError(f"position must be Sized. Got {pos}.")
        if len(pos) != 2:
            raise ValueError(f"Got position of length {len(pos)} != 2.")
        self.pos = np.array(pos, dtype=np.float32)
    
    @property
    def x(self):
        return self.pos[0]
    
    @property
    def y(self):
        return self.pos[1]

In [ ]:
v1, v2 = Vertex(), Vertex()
h1, h2 = HalfEdge(orig=v1, dest=v2), HalfEdge(orig=v2, dest=v1)
h1.rev = h2
h2.rev = h1
e = Edge(h1, h2)

In [ ]:
e[h1.orig]

In [ ]:
from copy import copy

v = Vertex()
d = dict()
d[v] = 3
v.any_outgoing = 123
d[v]

In [ ]:
from eucare.half import Vertex
from copy import deepcopy
a = Vertex()
a['a'] = a
b = deepcopy(a)
b['a']

In [ ]:
import cairo
width = 1000
height = 1000
surface = cairo.ImageSurface(
            cairo.FORMAT_RGB24, width, height)
dc = cairo.Context(surface)
dc.set_line_cap(cairo.LINE_CAP_ROUND)
dc.set_line_join(cairo.LINE_JOIN_ROUND)
dc.set_line_width(10)
dc.set_font_size(18.0)
dc.translate(width / 2, height / 2)
#dc.scale(self.scale, self.scale)
dc.set_source_rgb(0, 0, 0)
dc.paint()
surface.show_page()
surface.write_to_png('output.png')

from IPython.display import Image
Image(filename='output.png')

In [ ]:
{}

In [ ]:
import networkx as nx

g = nx.Graph
g.